In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                                DoubleType, DateType, TimestampType)

VOL = "/Volumes/workspace/default/maplebank"

customer_schema = StructType([
    StructField("customer_id",    StringType(), False),
    StructField("first_name",     StringType(), True),
    StructField("last_name",      StringType(), True),
    StructField("sin",            StringType(), True),   # PII — string, never number
    StructField("date_of_birth",  DateType(),   True),
    StructField("email",          StringType(), True),
    StructField("phone",          StringType(), True),
    StructField("street_address", StringType(), True),
    StructField("city",           StringType(), True),
    StructField("province",       StringType(), True),
    StructField("postal_code",    StringType(), True),
    StructField("customer_since", DateType(),   True),
])

account_schema = StructType([
    StructField("account_id",         StringType(), False),
    StructField("customer_id",        StringType(), False),
    StructField("account_number",     StringType(), True),  # String → keeps leading zeros!
    StructField("account_type",       StringType(), True),
    StructField("transit_number",     StringType(), True),
    StructField("institution_number", StringType(), True),
    StructField("open_date",          DateType(),   True),
    StructField("balance_cad",        DoubleType(), True),
    StructField("branch_id",          StringType(), True),
])

branch_schema = StructType([
    StructField("branch_id",      StringType(), False),
    StructField("branch_name",    StringType(), True),
    StructField("city",           StringType(), True),
    StructField("province",       StringType(), True),
    StructField("transit_number", StringType(), True),
])

txn_schema = StructType([
    StructField("transaction_id",        StringType(),    False),
    StructField("customer_id",           StringType(),    False),
    StructField("account_id",            StringType(),    False),
    StructField("branch_id",             StringType(),    True),
    StructField("transaction_date",      DateType(),      False),
    StructField("transaction_timestamp", TimestampType(), False),
    StructField("amount_cad",            DoubleType(),    False),
    StructField("transaction_type",      StringType(),    False),
    StructField("merchant_name",         StringType(),    True),
    StructField("channel",               StringType(),    True),
])

print("Schemas defined ✔")

In [0]:
df_customer = spark.read.option("header", True).schema(customer_schema).csv(f"{VOL}/dim_customer.csv")
df_account  = spark.read.option("header", True).schema(account_schema).csv(f"{VOL}/dim_account.csv")
df_branch   = spark.read.option("header", True).schema(branch_schema).csv(f"{VOL}/dim_branch.csv")
df_txn      = spark.read.option("header", True).schema(txn_schema).csv(f"{VOL}/fact_transactions.csv")

for name, df in [("customers", df_customer), ("accounts", df_account),
                 ("branches", df_branch), ("transactions", df_txn)]:
    print(f"{name:<14} {df.count():>7,} rows   {len(df.columns)} columns")

In [0]:
df_txn.printSchema()
df_txn.show(5, truncate=False)

In [0]:
def null_profile(df, name):
    print(f"\n── Null profile: {name} ──")
    df.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]).show(vertical=True)

null_profile(df_txn, "transactions")
null_profile(df_customer, "customers")

In [0]:
for name, df, key in [("transactions", df_txn, "transaction_id"),
                      ("customers", df_customer, "customer_id"),
                      ("accounts", df_account, "account_id")]:
    total = df.count()
    distinct = df.select(key).distinct().count()
    status = "✅ clean" if total == distinct else f"⚠ {total - distinct} duplicates!"
    print(f"{name:<14} total={total:>7,}  distinct {key}={distinct:>7,}  {status}")

In [0]:
# Transaction mix — what types flow through MapleBank?
df_txn.groupBy("transaction_type") \
      .agg(F.count("*").alias("count"), F.round(F.sum("amount_cad"), 2).alias("total_cad")) \
      .orderBy(F.col("total_cad").desc()) \
      .show()

# How many FINTRAC-territory transactions (≥ $10K)?
big = df_txn.filter(F.col("amount_cad") >= 10000).count()
print(f"Transactions ≥ $10,000 CAD: {big} ({big/df_txn.count()*100:.1f}%)")